# xx - Diagnostic Analytics

#### Objective

This notebook performs formal statistical analysis to investigate whether
flight-delay outcomes differ significantly across airlines, time periods,
airports, and other operational categories.

The analysis extends the descriptive findings from the exploratory data
analysis notebook by evaluating both statistical significance and practical
significance.

The notebook includes:

- Chi-square tests of independence
- Cramér's V effect-size measurement
- Kruskal-Wallis tests
- Post-hoc comparisons where appropriate
- Correlation analysis
- Practical interpretation of statistically significant results

#### Load and validate the cleaned dataset

This section loads the cleaned flight dataset produced by the data-cleaning
notebook. Diagnostic testing is performed only on cleaned records to ensure
that invalid, incomplete, and inconsistent observations do not affect the
statistical results.

Operated flights exclude cancelled and diverted records because arrival-delay
outcomes are not consistently available for those flights.

In [0]:
from config import project_config as cfg
from pyspark.sql import functions as F

# Confirm that the cleaned table exists before continuing.
if not spark.catalog.tableExists(cfg.CLEAN_TABLE):
    raise RuntimeError(
        f"Cleaned table '{cfg.CLEAN_TABLE}' was not found. "
        "Run the data-cleaning notebook before diagnostic analytics."
    )

# Load the cleaned dataset.
df = spark.read.table(cfg.CLEAN_TABLE)

# Retain operated flights for arrival-delay analysis.
df_operated = df.filter(
    (F.col(cfg.CANCELLED_COLUMN) == 0)
    & (F.col(cfg.DIVERTED_COLUMN) == 0)
)

total_flights = df.count()
operated_flights = df_operated.count()

print(" " * 60)
print("DIAGNOSTIC ANALYTICS DATA SOURCE:")
print(" " * 60)
print(f"Cleaned table: {cfg.CLEAN_TABLE}")
print(f"Total cleaned flights: {total_flights:,}")
print(f"Operated flights: {operated_flights:,}")
print(f"Columns available: {len(df.columns)}")

#### Statistical helper functions

The following helper functions support the statistical analyses performed in
this notebook. They calculate commonly used measures such as Cramér's V for
effect size and provide reusable utilities for reporting statistical results.

Using helper functions improves readability, consistency, and maintainability
throughout the diagnostic analyses.

In [0]:
import numpy as np
import pandas as pd

from scipy.stats import (
    chi2_contingency,
    kruskal,
    spearmanr
)

In [0]:
def cramers_v(contingency_table):
    """
    Calculate Cramér's V effect size from a contingency table.
    """

    chi2, _, _, _ = chi2_contingency(contingency_table)

    n = contingency_table.to_numpy().sum()

    r, c = contingency_table.shape

    return np.sqrt(
        chi2 /
        (
            n *
            (min(r - 1, c - 1))
        )
    )

In [0]:
def interpret_cramers_v(value):
    """
    Interpret the magnitude of Cramér's V.
    """

    if value < 0.10:
        return "Negligible"

    if value < 0.30:
        return "Small"

    if value < 0.50:
        return "Moderate"

    return "Strong"

#### Chi-square test: Airline and arrival delay status

This analysis evaluates whether arrival delay status is statistically associated
with the operating airline.

The null hypothesis states that airline and arrival delay status are
independent. The alternative hypothesis states that an association exists
between the two variables.

A Chi-square test of independence is used to determine statistical
significance, while Cramér's V is reported to measure the strength of the
association.

In [0]:
airline_delay_table = (
    df_operated
    .groupBy(
        cfg.AIRLINE_COLUMN,
        cfg.ARRIVAL_DELAY_FLAG_COLUMN
    )
    .count()
    .toPandas()
    .pivot(
        index=cfg.AIRLINE_COLUMN,
        columns=cfg.ARRIVAL_DELAY_FLAG_COLUMN,
        values="count"
    )
    .fillna(0)
)

display(airline_delay_table.reset_index())

In [0]:
chi2, p_value, dof, expected = chi2_contingency(
    airline_delay_table
)

cramers = cramers_v(
    airline_delay_table
)

results = pd.DataFrame({
    "Statistic": [
        "Chi-square",
        "Degrees of Freedom",
        "P-value",
        "Cramer's V",
        "Association Strength"
    ],
    "Value": [
        f"{chi2:,.2f}",
        str(dof),
        f"{p_value:.6e}",
        f"{cramers:.3f}",
        interpret_cramers_v(cramers)
    ]
})

display(results)

alpha = 0.05

if p_value < alpha:
    significance_conclusion = (
        "Reject the null hypothesis. Airline and arrival delay status "
        "have a statistically significant association."
    )
else:
    significance_conclusion = (
        "Fail to reject the null hypothesis. There is insufficient evidence "
        "of an association between airline and arrival delay status."
    )

print(significance_conclusion)
print(
    f"Cramer's V = {cramers:.3f}, indicating a "
    f"{interpret_cramers_v(cramers).lower()} association."
)

#### Findings

The Chi-square test produced a statistically significant result, indicating
that arrival-delay status is associated with the operating airline.

However, Cramér's V was 0.059, which represents a negligible effect size.
Therefore, although delay rates differ across airlines, airline membership
alone has limited practical explanatory power for arrival-delay outcomes.

The very large sample size likely contributed to the extremely small p-value,
making even minor differences statistically significant. For this reason,
effect size should be considered alongside statistical significance.

#### Chi-square test: Month and arrival delay status

This analysis evaluates whether arrival-delay status is associated with the
month in which a flight operates.

The null hypothesis states that month and arrival-delay status are independent.
The alternative hypothesis states that delay outcomes vary across months.

A Chi-square test is used to evaluate statistical significance, while
Cramér's V measures the strength of the association.

In [0]:
month_delay_table = (
    df_operated
    .groupBy(
        cfg.MONTH_COLUMN,
        cfg.ARRIVAL_DELAY_FLAG_COLUMN
    )
    .count()
    .toPandas()
    .pivot(
        index=cfg.MONTH_COLUMN,
        columns=cfg.ARRIVAL_DELAY_FLAG_COLUMN,
        values="count"
    )
    .fillna(0)
    .sort_index()
)

display(month_delay_table.reset_index())

In [0]:
chi2, p_value, dof, expected = chi2_contingency(
    month_delay_table
)

cramers = cramers_v(
    month_delay_table
)

month_results = pd.DataFrame({
    "Statistic": [
        "Chi-square",
        "Degrees of Freedom",
        "P-value",
        "Cramer's V",
        "Association Strength"
    ],
    "Value": [
        f"{chi2:,.2f}",
        str(dof),
        f"{p_value:.6e}",
        f"{cramers:.3f}",
        interpret_cramers_v(cramers)
    ]
})

display(month_results)

alpha = 0.05

if p_value < alpha:
    significance_conclusion = (
        "Reject the null hypothesis. Month and arrival delay status "
        "have a statistically significant association."
    )
else:
    significance_conclusion = (
        "Fail to reject the null hypothesis. There is insufficient evidence "
        "of an association between month and arrival delay status."
    )

print(significance_conclusion)

print(
    f"Cramer's V = {cramers:.3f}, indicating a "
    f"{interpret_cramers_v(cramers).lower()} association."
)

#### Findings

The Chi-square test indicated a statistically significant association between
month of operation and arrival delay status (p < 0.001).

However, the corresponding Cramér's V value of 0.091 indicates a negligible
effect size. Although delay rates vary across months, the magnitude of these
differences is relatively small in practical terms.

The significant result is likely influenced by the very large sample size,
highlighting the importance of interpreting effect size alongside statistical
significance. Seasonal weather conditions, holiday travel demand, and airport
congestion may contribute to the observed monthly differences.

#### Kruskal–Wallis test: Arrival delay across airlines

This analysis evaluates whether the distribution of arrival delay minutes
differs among airlines.

Unlike the Chi-square test, which analyzes categorical outcomes, the
Kruskal–Wallis test compares the distributions of a continuous variable across
multiple independent groups.

The null hypothesis states that all airlines have the same distribution of
arrival delay minutes. The alternative hypothesis states that at least one
airline differs from the others.

The Kruskal–Wallis test is used because arrival-delay minutes are not normally
distributed and contain substantial skewness.

In [0]:
arrival_delay_by_airline = (
    df_operated
    .select(
        cfg.AIRLINE_COLUMN,
        cfg.ARRIVAL_DELAY_COLUMN
    )
    .dropna()
    .toPandas()
)

arrival_delay_by_airline.head()

In [0]:
# Create one delay array for each airline
airline_groups = [
    group["ARR_DELAY"].values
    for _, group in arrival_delay_by_airline.groupby(cfg.AIRLINE_COLUMN)
]

# Perform the Kruskal-Wallis test
h_statistic, p_value = kruskal(*airline_groups)

print(f"Kruskal-Wallis H statistic : {h_statistic:,.2f}")

if p_value < 0.001:
    print("P-value                   : < 0.001")
else:
    print(f"P-value                   : {p_value:.4f}")

In [0]:
kruskal_results = pd.DataFrame({
    "Statistic": [
        "Kruskal-Wallis H",
        "P-value"
    ],
    "Value": [
        f"{h_statistic:,.2f}",
        "< 0.001" if p_value < 0.001 else f"{p_value:.4f}"
    ]
})

display(kruskal_results)

alpha = 0.05

if p_value < alpha:
    print(
        "Reject the null hypothesis. "
        "At least one airline has a significantly different "
        "arrival-delay distribution."
    )
else:
    print(
        "Fail to reject the null hypothesis. "
        "There is insufficient evidence that airlines differ "
        "in their arrival-delay distributions."
    )

#### Findings

The Kruskal–Wallis test identified a statistically significant difference in
arrival-delay distributions across airlines, H = 40,712.98, p < 0.001.

Therefore, the null hypothesis was rejected. This indicates that at least one
airline has an arrival-delay distribution that differs significantly from the
others.

However, the Kruskal–Wallis test does not identify which specific airlines
differ. A post-hoc pairwise comparison is therefore required to determine where
the significant differences occur.

#### Dunn's post-hoc pairwise comparisons

Because the Kruskal–Wallis test identified a statistically significant
difference in arrival-delay distributions across airlines, a post-hoc analysis
was performed to determine which airline pairs differ significantly.

Dunn's test with Bonferroni correction was used to control the family-wise
error rate associated with multiple pairwise comparisons.

Adjusted p-values below 0.05 indicate statistically significant differences
between airline pairs.

In [0]:
import scikit_posthocs as sp

dunn_results = sp.posthoc_dunn(
    arrival_delay_by_airline,
    val_col=cfg.ARRIVAL_DELAY_COLUMN,
    group_col=cfg.AIRLINE_COLUMN,
    p_adjust="bonferroni"
)

display(dunn_results)

In [0]:
significant_pairs = []

airlines = dunn_results.index.tolist()

for i in range(len(airlines)):
    for j in range(i + 1, len(airlines)):
        p = dunn_results.iloc[i, j]

        if p < 0.05:
            significant_pairs.append({
                "Airline 1": airlines[i],
                "Airline 2": airlines[j],
                "Adjusted P-value": round(p, 6)
            })

significant_pairs = []

airlines = dunn_results.index.tolist()

for i in range(len(airlines)):
    for j in range(i + 1, len(airlines)):
        p = dunn_results.iloc[i, j]

        if p < 0.05:
            significant_pairs.append({
                "Airline 1": airlines[i],
                "Airline 2": airlines[j],
                "Adjusted P-value": (
                    "< 0.001"
                    if p < 0.001
                    else f"{p:.4f}"
                )
            })

significant_pairs = (
    pd.DataFrame(significant_pairs)
    .sort_values("Adjusted P-value")
)

display(significant_pairs)

#### Findings

Following the significant Kruskal–Wallis result, Dunn's post-hoc test with
Bonferroni correction was performed to identify the airline pairs with
statistically significant differences in arrival-delay distributions.

The analysis identified numerous airline pairs with adjusted p-values below
0.05, indicating that their arrival-delay distributions differ significantly.

Given the very large sample size used in this study, many pairwise comparisons
were statistically significant after adjustment for multiple testing. Therefore,
these findings should be interpreted alongside practical significance and
operational relevance rather than statistical significance alone.

#### Spearman correlation analysis

This analysis examines the relationships among selected numerical operational
variables using Spearman's rank correlation coefficient.

Unlike Pearson correlation, Spearman correlation does not assume normally
distributed data and is therefore appropriate for the highly skewed
distribution of flight-delay variables.

Correlation coefficients range from -1 to +1, where values closer to ±1
indicate stronger relationships and values near zero indicate weak or no
monotonic relationship.

In [0]:
correlation_columns = [
    cfg.DEPARTURE_DELAY_COLUMN,
    cfg.TAXI_OUT_COLUMN,
    cfg.TAXI_IN_COLUMN,
    cfg.AIR_TIME_COLUMN,
    cfg.DISTANCE_COLUMN,
    cfg.ARRIVAL_DELAY_COLUMN
]

correlation_df = (
    df_operated
    .select(correlation_columns)
    .dropna()
    .toPandas()
)

correlation_df.head()

In [0]:
spearman_corr = correlation_df.corr(method="spearman")

display(spearman_corr.round(3))

#### Visualization

The following heatmap illustrates the Spearman correlation coefficients among
selected operational variables.

Values closer to +1 indicate stronger positive relationships, whereas values
closer to -1 indicate stronger negative relationships. Values near zero
indicate little or no monotonic relationship.

In [0]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))

image = plt.imshow(
    spearman_corr,
    interpolation="nearest"
)

plt.colorbar(image, label="Spearman Correlation")

plt.xticks(
    range(len(spearman_corr.columns)),
    spearman_corr.columns,
    rotation=45,
    ha="right"
)

plt.yticks(
    range(len(spearman_corr.index)),
    spearman_corr.index
)

# Display the correlation values inside each cell
for i in range(len(spearman_corr.index)):
    for j in range(len(spearman_corr.columns)):
        plt.text(
            j,
            i,
            f"{spearman_corr.iloc[i, j]:.2f}",
            ha="center",
            va="center",
            fontsize=9
        )

plt.title("Spearman Correlation Matrix")

plt.tight_layout()

plt.show()

In [0]:
correlation_summary = (
    spearman_corr["ARR_DELAY"]
    .drop("ARR_DELAY")
    .abs()
    .sort_values(ascending=False)
    .reset_index()
)

correlation_summary.columns = [
    "Operational Variable",
    "Absolute Spearman Correlation"
]

display(correlation_summary)

#### Findings

The Spearman correlation analysis identified the relationships among selected
operational variables associated with arrival delay.

Departure delay exhibited the strongest positive relationship with arrival
delay (ρ = 0.707), indicating that flights departing late are considerably
more likely to arrive late.

Taxi-out time showed a weak positive association (ρ = 0.292), suggesting that
extended ground movement before takeoff contributes modestly to arrival delays.
Taxi-in time demonstrated only a weak relationship (ρ = 0.109).

Air time (ρ = 0.015) and flight distance (ρ = −0.018) showed negligible
relationships with arrival delay, indicating that flight duration and travel
distance alone are poor indicators of arrival punctuality.

A very strong positive correlation was observed between air time and flight
distance (ρ = 0.986), which is expected because longer flights generally cover
greater distances.

## Diagnostic Analytics Summary

The diagnostic analyses provided statistical evidence regarding the factors
associated with flight arrival delays.

Chi-square tests demonstrated statistically significant associations between
arrival delay status and both airline and month of operation. However,
Cramér's V indicated negligible effect sizes, suggesting that these categorical
variables alone have limited practical influence despite statistical
significance.

The Kruskal–Wallis test identified significant differences in arrival-delay
distributions across airlines. Dunn's post-hoc analysis further confirmed that
multiple airline pairs differed significantly after applying the Bonferroni
correction for multiple comparisons.

Spearman correlation analysis revealed that departure delay exhibited the
strongest positive relationship with arrival delay, while taxi-out time showed
a weaker positive association. Flight distance and air time demonstrated
minimal direct relationships with arrival delay, although they were strongly
correlated with each other.

Overall, the diagnostic analyses indicate that pre-departure operational
conditions are more strongly associated with arrival punctuality than flight
characteristics such as distance or air time. These findings provide valuable
insights for feature engineering and support the development of predictive
models in the subsequent stages of the project.

In [0]:
print("Diagnostic Analysis completed successfully.")